In [1]:
import bigfile
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from mpl_toolkits.mplot3d import Axes3D
from nbodykit.lab import BigFileCatalog
from scipy.spatial import distance

# Import class to analyze MP_Gadget run snapshots
from dm_snapshot_analysis import SnapshotData

In [3]:
# Snapshot imports to obtain DM particle attributes
snapshot000 = '/home/aschann/MP-Gadget/DM_Halos_April_7/output/PART_000'
snapshot001 = '/home/aschann/MP-Gadget/DM_Halos_April_7/output/PART_001'
snapshot002 = '/home/aschann/MP-Gadget/DM_Halos_April_7/output/PART_002'
snapshot003 = '/home/aschann/MP-Gadget/DM_Halos_April_7/output/PART_003'
snapshot004 = '/home/aschann/MP-Gadget/DM_Halos_April_7/output/PART_004'
snapshot005 = '/home/aschann/MP-Gadget/DM_Halos_April_7/output/PART_005'
snapshot006 = '/home/aschann/MP-Gadget/DM_Halos_April_7/output/PART_006'

# Import class to obtain data from snapshot 
snap0 = SnapshotData(snapshot000)
snap1 = SnapshotData(snapshot001)
snap2 = SnapshotData(snapshot002)
snap3 = SnapshotData(snapshot003)
snap4 = SnapshotData(snapshot004)
snap5 = SnapshotData(snapshot005)
snap6 = SnapshotData(snapshot006)

/home/aschann/miniconda/envs/test/lib/python3.6/site-packages/bigfile/__init__.py:358: FutureWarning: Passing (type, 1) or '1type' as a synonym of type is deprecated; in a future version of numpy, it will be understood as (type, (1,)) / '(1,)type'.
  return pyxbigfile.Dataset.__init__(self, file, dtype=dtype, size=size)


In [4]:
# Import data manually/separately to look at 'new' parameters
data = BigFileCatalog(snapshot000, dataset='1/', header='Header')

pos = data['Position'].compute() / 1000

v = data['Velocity'].compute()
u = data['Potential'].compute()
m = data['Mass'].compute()

m = m[0] # Masses are the same for every particle
U = m * u # U = m * Grav. Potential

In [5]:
m

465.52756

In [6]:
v

array([[ 22.63697 , -57.42381 ,  61.323414],
       [ 22.3273  , -48.140186,  50.512665],
       [-10.405549, -55.466026,  68.42081 ],
       ...,
       [-24.50619 , -42.92058 ,  71.414986],
       [  8.012158, -55.06678 ,  71.98856 ],
       [  6.408032, -31.817816,  84.79968 ]], dtype=float32)

In [7]:
(m *1.989e43) / 1.989e33 # converting from internal units to grams to solar masses

4655275573730.469

In [8]:
m*1e10 # same thing as above

4655275573730.469

In [9]:
m*1.989e43

9.259343116149903e+45

In [10]:
POS = data['Position'].compute() 
POS 

array([[496157.3772212 , 495941.29155994, 496258.02933271],
       [496154.75915804, 499880.79727647, 496227.07621221],
       [499967.33240329, 499851.08620775, 496279.32275383],
       ...,
       [499925.221803  , 499879.98637682, 492371.86204902],
       [496115.29510515, 499837.68699276, 492385.57634917],
       [496110.39011969, 496011.636946  , 492415.81009377]])

In [11]:
pos

array([[496.15737722, 495.94129156, 496.25802933],
       [496.15475916, 499.88079728, 496.22707621],
       [499.9673324 , 499.85108621, 496.27932275],
       ...,
       [499.9252218 , 499.87998638, 492.37186205],
       [496.11529511, 499.83768699, 492.38557635],
       [496.11039012, 496.01163695, 492.41581009]])

In [12]:
u[0] # gravitational potential of first particle

-22279.596

In [13]:
### Check potential energy for ONE particle to confirm units

def calculate_potential_at_particle(index, positions, masses, G=6.67430e-8):
    """
    Calculate the gravitational potential at the location of a specific particle.
    
    Parameters:
    - index: index of the particle in the arrays
    - positions: numpy array of shape (N, 3) containing the positions of N particles
    - masses: numpy array of shape (N,) containing the masses of N particles
    - G: gravitational constant in cgs units (cm^3 g^-1 s^-2)
    
    Returns:
    - potential: gravitational potential at the particle due to all other particles
    """
    # Extract the position of the target particle
    target_pos = positions[index]
    
    # Calculate the vector distances from the target particle to all others
    r_vectors = positions - target_pos
    
    # Calculate the magnitudes of these distance vectors
    distances = np.linalg.norm(r_vectors, axis=1)
    
    # Avoid division by zero by setting the distance at the index to infinity
    distances[index] = np.inf
    
    # Calculate potential contributions from each particle
    potential_contributions = -G * masses / distances
    
    # Sum up contributions to get total potential at the particle
    total_potential = np.sum(potential_contributions)
    
    return total_potential


calculate_potential_at_particle(0, pos, m)  

-0.15760498134954054

In [14]:
U

array([-10371766., -10422466., -10417778., ..., -10483499., -10554238.,
       -10542187.], dtype=float32)

In [15]:
v_squared = np.sum((v**2), axis=1)

T = 0.5* v_squared * m

Etot = U + T  # total energy of each particle
Etot

array([-8609631., -9173103., -8586820., ..., -8727801., -8627210.,
       -8623184.], dtype=float32)

In [16]:
np.sum(Etot)  # total energy of system

-17083367000000.0

### None of this mumbo jumbo makes sense

In [17]:
# Let's look at just the gravitational potential energies given.

UG0 = (BigFileCatalog(snapshot000, dataset='1/', header='Header'))['Potential'].compute()
UG1 = (BigFileCatalog(snapshot001, dataset='1/', header='Header'))['Potential'].compute()
UG2 = (BigFileCatalog(snapshot002, dataset='1/', header='Header'))['Potential'].compute()
UG3 = (BigFileCatalog(snapshot003, dataset='1/', header='Header'))['Potential'].compute()
UG4 = (BigFileCatalog(snapshot004, dataset='1/', header='Header'))['Potential'].compute()
UG5 = (BigFileCatalog(snapshot005, dataset='1/', header='Header'))['Potential'].compute()
UG6 = (BigFileCatalog(snapshot006, dataset='1/', header='Header'))['Potential'].compute()

v0 = (BigFileCatalog(snapshot000, dataset='1/', header='Header'))['Velocity'].compute()
v1 = (BigFileCatalog(snapshot000, dataset='1/', header='Header'))['Velocity'].compute()
v2 = (BigFileCatalog(snapshot000, dataset='1/', header='Header'))['Velocity'].compute()
v3 = (BigFileCatalog(snapshot000, dataset='1/', header='Header'))['Velocity'].compute()
v4 = (BigFileCatalog(snapshot000, dataset='1/', header='Header'))['Velocity'].compute()
v5 = (BigFileCatalog(snapshot000, dataset='1/', header='Header'))['Velocity'].compute()
v6 = (BigFileCatalog(snapshot000, dataset='1/', header='Header'))['Velocity'].compute()

m = (BigFileCatalog(snapshot000, dataset='1/', header='Header'))['Mass'].compute()

m = m[0] # Masses are the same for every particle
# U = m * Grav. Potential
U0 = m * UG0 
U1 = m * UG1
U2 = m * UG2 
U3 = m * UG3 
U4 = m * UG4 
U5 = m * UG5 
U6 = m * UG6 

In [18]:
# Total Gravitational Potential Energies

UG0_tot = np.sum(UG0)
UG1_tot = np.sum(UG1)
UG2_tot = np.sum(UG2)
UG3_tot = np.sum(UG3)
UG4_tot = np.sum(UG4)
UG5_tot = np.sum(UG5)
UG6_tot = np.sum(UG6)

UG0_tot = format(UG0_tot, ".2e")
UG1_tot = format(UG1_tot, ".2e")
UG2_tot = format(UG2_tot, ".2e")
UG3_tot = format(UG3_tot, ".2e")
UG4_tot = format(UG4_tot, ".2e")
UG5_tot = format(UG5_tot, ".2e")
UG6_tot = format(UG6_tot, ".2e")

In [19]:
# Total system energies

Etot0 = 0.5*(np.sum((v0**2), axis=1))*m + U0 
Etot1 = 0.5*(np.sum((v1**2), axis=1))*m + U1 
Etot2 = 0.5*(np.sum((v2**2), axis=1))*m + U2 
Etot3 = 0.5*(np.sum((v3**2), axis=1))*m + U3 
Etot4 = 0.5*(np.sum((v4**2), axis=1))*m + U4 
Etot5 = 0.5*(np.sum((v5**2), axis=1))*m + U5 
Etot6 = 0.5*(np.sum((v6**2), axis=1))*m + U6 
             
E0 = np.sum(Etot0)
E1 = np.sum(Etot1)
E2 = np.sum(Etot2)
E3 = np.sum(Etot3)            
E4 = np.sum(Etot4)             
E5 = np.sum(Etot5)            
E6 = np.sum(Etot6)           
             
E0 = format(E0, ".2e")
E1 = format(E1, ".2e")
E2 = format(E2, ".2e")
E3 = format(E3, ".2e")
E4 = format(E4, ".2e")
E5 = format(E5, ".2e")
E6 = format(E6, ".2e")

In [20]:
# Total Gravitational Potential Energies

print(UG0_tot , "\n",
      UG1_tot , "\n",
      UG2_tot , "\n",
      UG3_tot , "\n",
      UG4_tot , "\n",
      UG5_tot , "\n",
      UG6_tot , "\n",
     )  # Grav potential energy increases (in magnitude, becomes more negative)

-4.66e+10 
 -4.75e+10 
 -4.88e+10 
 -5.08e+10 
 -6.58e+10 
 -1.13e+11 
 -3.79e+11 



In [21]:
# Total system energies

print(E0 , "\n",
      E1 , "\n",
      E2 , "\n",
      E3 , "\n",
      E4 , "\n",
      E5 , "\n",
      E6 , "\n",
     )  # Total energy increases 

-1.71e+13 
 -1.75e+13 
 -1.81e+13 
 -1.91e+13 
 -2.60e+13 
 -4.82e+13 
 -1.72e+14 



In [22]:
### Second check using E = mc**2 + 1/2mv**2
c = 3e10  # speed of light in cm/s

"""Etot0_2 = 0.5*(np.sum((v0**2), axis=1))*m + m*c**2
Etot1_2 = 0.5*(np.sum((v1**2), axis=1))*m + m*c**2
Etot2_2 = 0.5*(np.sum((v2**2), axis=1))*m + m*c**2
Etot3_2 = 0.5*(np.sum((v3**2), axis=1))*m + m*c**2
Etot4_2 = 0.5*(np.sum((v4**2), axis=1))*m + m*c**2
Etot5_2 = 0.5*(np.sum((v5**2), axis=1))*m + m*c**2
Etot6_2 = 0.5*(np.sum((v6**2), axis=1))*m + m*c**2"""

Etot0_2 = m*c**2
Etot1_2 = m*c**2
Etot2_2 = m*c**2
Etot3_2 = m*c**2
Etot4_2 = m*c**2
Etot5_2 = m*c**2
Etot6_2 = m*c**2

             
E0_2 = np.sum(Etot0_2)
E1_2 = np.sum(Etot1_2)
E2_2 = np.sum(Etot2_2)
E3_2 = np.sum(Etot3_2)            
E4_2 = np.sum(Etot4_2)             
E5_2 = np.sum(Etot5_2)            
E6_2 = np.sum(Etot6_2)           
             
E0_2 = format(E0_2, ".2e")
E1_2 = format(E1_2, ".2e")
E2_2 = format(E2_2, ".2e")
E3_2 = format(E3_2, ".2e")
E4_2 = format(E4_2, ".2e")
E5_2 = format(E5_2, ".2e")
E6_2 = format(E6_2, ".2e")

In [23]:
# Total system energies

print(E0_2 , "\n",
      E1_2 , "\n",
      E2_2 , "\n",
      E3_2 , "\n",
      E4_2 , "\n",
      E5_2 , "\n",
      E6_2 , "\n",
     )  

4.19e+23 
 4.19e+23 
 4.19e+23 
 4.19e+23 
 4.19e+23 
 4.19e+23 
 4.19e+23 

